<a href="https://colab.research.google.com/github/Suraj-Sedai/Hardware-Aware-LLM-Inference-Engine/blob/main/notebooks/00_overall_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [130]:
import torch
from torch import nn
from typing import Optional, Tuple


In [131]:
class KVCacheManager:
    def __init__(self, n_layers, n_heads, max_seq_len, dim_head, device, batch_size):
        self.n_layers = n_layers
        self.n_heads = n_heads
        self.max_seq_len = max_seq_len
        self.dim_head = dim_head
        self.device = device
        self.batch_size = batch_size

        # preallocate KV cache for all layers, now with batch_size
        self.K = [torch.zeros(batch_size, n_heads, max_seq_len, dim_head, device=device) for _ in range(n_layers)]
        self.V = [torch.zeros(batch_size, n_heads, max_seq_len, dim_head, device=device) for _ in range(n_layers)]
        self.curr_len = 0  # tokens stored

    def update(self, layer_id, K_new, V_new):
        # K_new, V_new are expected to be (B, n_heads, T_new, dim_head)
        B, num_heads, T_new, dim_head = K_new.shape
        assert B == self.batch_size
        assert num_heads == self.n_heads
        assert dim_head == self.dim_head

        # Ensure there's space in the cache
        if self.curr_len + T_new > self.max_seq_len:
            raise ValueError("KV cache out of capacity!")

        # Update the cache for the current layer and for the new tokens
        self.K[layer_id][:, :, self.curr_len : self.curr_len + T_new, :] = K_new
        self.V[layer_id][:, :, self.curr_len : self.curr_len + T_new, :] = V_new

        # curr_len is updated once per GPT.forward call, not per layer update

    def get(self, layer_id):
        return self.K[layer_id][:, :, :self.curr_len, :], self.V[layer_id][:, :, :self.curr_len, :]


In [132]:
class SelfAttention(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

    def forward(self, x, kv_cache=None, layer_id=None):
        B, T, D = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # reshape for heads
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2) # (B, n_heads, T, head_dim)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2) # (B, n_heads, T, head_dim)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2) # (B, n_heads, T, head_dim)

        if kv_cache is not None and kv_cache.curr_len > 0: # Only retrieve from cache if it's not empty
            K_cached, V_cached = kv_cache.get(layer_id) # K_cached: (B, n_heads, curr_len, head_dim)
            K_all = torch.cat([K_cached, k], dim=2) # Concatenate along sequence length dimension (dim=2)
            V_all = torch.cat([V_cached, v], dim=2)
        else:
            K_all, V_all = k, v # K_all, V_all are just the new k, v

        # attention computation
        attn = (q @ K_all.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = attn @ V_all

        # merge heads
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out), k, v # k, v here are the new tokens' k, v (B, n_heads, T, head_head)


In [133]:
class MLP(nn.Module):
    def __init__(self, dim, hidden_dim=None):
        super().__init__()
        hidden_dim = hidden_dim or dim * 4
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

In [134]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.attn = SelfAttention(dim, n_heads)
        self.mlp = MLP(dim)

    def forward(self, x, kv_cache=None, layer_id=None):
        # Attention block
        attn_out, K_new, V_new = self.attn(self.ln1(x), kv_cache, layer_id)
        x = x + attn_out
        # MLP block
        x = x + self.mlp(self.ln2(x))
        return x, K_new, V_new

In [135]:
class GPT(nn.Module):
    def __init__(self, vocab_size, dim, n_heads, n_layers, max_seq_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_seq_len, dim)
        self.blocks = nn.ModuleList([TransformerBlock(dim, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_size)
        self.max_seq_len = max_seq_len

    def forward(self, input_ids, kv_cache=None):
        B, T = input_ids.shape

        if kv_cache is not None and kv_cache.curr_len > 0:
            # During generation, input_ids is typically (B, 1)
            # The position for this token is the current length of the cache
            pos_start_idx = kv_cache.curr_len
        else:
            # During prefill, input_ids is the full prompt (B, T)
            # The positions are 0 to T-1
            pos_start_idx = 0

        # Create positional embeddings for the current input_ids
        # These positions correspond to the *absolute* positions in the sequence
        positions = torch.arange(pos_start_idx, pos_start_idx + T, device=input_ids.device).unsqueeze(0) # (1, T)
        positions = positions.repeat(B, 1) # Make it (B, T) to match input_ids batch

        x = self.token_emb(input_ids) + self.pos_emb(positions)

        # List to store new K,V for all layers in this forward pass
        new_ks_vs = []

        for layer_id, block in enumerate(self.blocks):
            x, K_new, V_new = block(x, kv_cache, layer_id) # K_new, V_new are (B, n_heads, T, dim_head)
            new_ks_vs.append((K_new, V_new))

        # Update kv_cache after all blocks have processed the current input
        if kv_cache is not None:
            for layer_id, (K_new, V_new) in enumerate(new_ks_vs):
                kv_cache.update(layer_id, K_new, V_new)
            kv_cache.curr_len += T # Update cache length based on new tokens processed

        x = self.ln_f(x)
        logits = self.head(x)
        return logits


In [155]:
def sample_top_k(logits, k=50, temperature=1.0):
    logits = logits / temperature
    # Ensure k does not exceed the vocabulary size
    k = min(k, logits.size(-1))
    top_k_vals, top_k_idx = torch.topk(logits, k)
    probs = torch.softmax(top_k_vals, dim=-1)
    idx = torch.multinomial(probs, 1)
    return top_k_idx.gather(-1, idx)

In [137]:
def generate(model, input_ids, kv_cache, max_new_tokens):
    generated = input_ids.clone()
    for step in range(max_new_tokens):
        # last token only
        x = generated[:, -1:]
        logits = model(x, kv_cache)
        next_token = sample_top_k(logits[:, -1, :])
        generated = torch.cat([generated, next_token.unsqueeze(-1)], dim=1)
    return generated

##Minimal Test Case

In [138]:
import torch
import torch.nn as nn

# Tiny model for testing
vocab_size = 100
dim = 32
n_heads = 4
n_layers = 2
max_seq_len = 16
device = "cuda" if torch.cuda.is_available() else "cpu"

# Assuming batch_size is 1 for this test case based on input_ids later
batch_size = 1

# KV Cache
kv_cache = KVCacheManager(n_layers, n_heads, max_seq_len, dim // n_heads, device, batch_size)

# Model
model = GPT(vocab_size, dim, n_heads, n_layers, max_seq_len).to(device)


In [139]:
# Batch size 1, prompt length 4
input_ids = torch.tensor([[10, 25, 30, 7]], device=device)

In [140]:
# Prefill: compute over full prompt
logits = model(input_ids, kv_cache=kv_cache)
print("Prefill logits shape:", logits.shape)  # Expected: [1, 4, 100]

Prefill logits shape: torch.Size([1, 4, 100])


## text generation with tokenizer

In [141]:
# simple tokenizer from nb 01
class SimpleTokenizer:
    def __init__(self, vocab):
        self.vocab = vocab
        self.inv_vocab = {i: t for t, i in vocab.items()}

    def encode(self, text):
        return [self.vocab.get(tok, self.vocab['<UNK>']) for tok in text.split()]

    def decode(self, token_ids):
        return ' '.join([self.inv_vocab.get(i, '<UNK>') for i in token_ids])

In [142]:
# make a tiny vocab for demo
from collections import Counter

corpus = "hello world this is a test hello test world"
words = corpus.split()
counts = Counter(words)

vocab = {'<PAD>': 0, '<UNK>': 1}
for word, _ in counts.most_common(100):
    vocab[word] = len(vocab)

print(f"vocab size: {len(vocab)}")
print(f"vocab: {vocab}")

vocab size: 8
vocab: {'<PAD>': 0, '<UNK>': 1, 'hello': 2, 'world': 3, 'test': 4, 'this': 5, 'is': 6, 'a': 7}


In [143]:
# init tokenizer
tokenizer = SimpleTokenizer(vocab)

# test it
text = "hello world"
ids = tokenizer.encode(text)
print(f"encode '{text}': {ids}")
print(f"decode {ids}: '{tokenizer.decode(ids)}'")

encode 'hello world': [2, 3]
decode [2, 3]: 'hello world'


In [144]:
# rebuild model with new vocab size
vocab_size = len(vocab)
dim = 32
n_heads = 4
n_layers = 2
max_seq_len = 16
batch_size = 1

model = GPT(vocab_size, dim, n_heads, n_layers, max_seq_len).to(device)
kv_cache = KVCacheManager(n_layers, n_heads, max_seq_len, dim // n_heads, device, batch_size)

print(f"model ready, vocab_size={vocab_size}")

model ready, vocab_size=8


In [145]:
# text in -> model -> text out
input_text = "hello world"
print(f"input: {input_text}")

# encode
input_ids = torch.tensor([tokenizer.encode(input_text)], device=device)
print(f"token ids: {input_ids}")

# generate
generated = input_ids.clone()
for step in range(3):
    x = generated[:, -1:]
    logits = model(x, kv_cache=kv_cache)
    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
    generated = torch.cat([generated, next_token], dim=1)

# decode
output_text = tokenizer.decode(generated[0].tolist())
print(f"\noutput: {output_text}")
print("\nnote: model is untrained so output is random")

input: hello world
token ids: tensor([[2, 3]])

output: hello world a a <UNK>

note: model is untrained so output is random


In [146]:
# Start with last token from prompt
generated = input_ids.clone()
for step in range(3):  # generate 3 new tokens
    x = generated[:, -1:]
    logits = model(x, kv_cache=kv_cache)
    # simple argmax sampling
    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
    generated = torch.cat([generated, next_token], dim=1)

print("Generated sequence:", generated)

Generated sequence: tensor([[2, 3, 1, 1, 1]])


###Profiling

In [157]:
import time

def benchmark(model, input_ids, kv_cache, max_new_tokens):
    # Removed torch.cuda.synchronize() as device is CPU

    start = time.time()

    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        x = generated[:, -1:]
        logits = model(x, kv_cache)
        next_token = sample_top_k(logits[:, -1, :])
        generated = torch.cat([generated, next_token], dim=1)

    # Removed torch.cuda.synchronize() as device is CPU
    end = time.time()

    total_time = end - start
    tokens_generated = max_new_tokens

    print(f"Total time: {total_time:.4f}s")
    print(f"Tokens/sec: {tokens_generated / total_time:.2f}")
    print(f"Latency per token: {total_time / tokens_generated:.4f}s")


In [158]:
# Re-initialize KV cache for a clean benchmark run
kv_cache = KVCacheManager(n_layers, n_heads, max_seq_len, dim // n_heads, device, batch_size)

# Define the number of new tokens to generate for benchmarking
max_new_tokens_benchmark = 10

print(f"Benchmarking generation of {max_new_tokens_benchmark} new tokens...")
benchmark(model, input_ids, kv_cache, max_new_tokens_benchmark)

Benchmarking generation of 10 new tokens...
Total time: 0.0235s
Tokens/sec: 425.36
Latency per token: 0.0024s


In [150]:
import torch.profiler

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU], # Changed to CPU
    record_shapes=True
) as prof:

    logits = model(input_ids, kv_cache)

print(prof.key_averages().table(sort_by="cpu_time_total")) # Changed sort_by

---------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                       Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
---------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
               aten::arange        23.20%       3.024ms        30.55%       3.980ms     995.119us             4  
               aten::repeat        27.06%       3.526ms        28.87%       3.761ms       3.761ms             1  
           aten::layer_norm        21.00%       2.736ms        22.54%       2.938ms     587.500us             5  
              aten::softmax         7.21%     939.344us         7.52%     980.496us     490.248us             2  
               aten::linear         0.85%     110.532us         6.73%     876.754us      67.443us            13  
                aten::addmm         2.40%     312.947us         3.33%     433.389us     

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
